# Library

In [1]:
from abc import ABC, abstractmethod
from datetime import datetime

import sqlite3

# Constant

In [ ]:
DBWRITER_TYPE_SQLITE = 'db_sqlite'
DBWRITER_TYPE_TSV    = 'tsv'

DB_PATH_DEVICE_SQLITE = './data/iot_raw_database.sqlite'
DB_PATH_DEVICE_TSV    = './data/iot_raw_database.tsv'

DB_TBL_NAME_RAW_DATA = 'tbl_raw_data_in'

# DBWriter

## DBWriter base

In [3]:
class DBWriter(ABC):    
    def __init__(self, aDevice):
        self._batch_sn = 0     

    def _combine_fields(self, aUId, aPId, aTick, aType, aData):
        return ["'{tp_srv_in}'"\
                    .format(tp_srv_in = datetime.now()\
                    .strftime("%Y-%m-%d %H:%M:%S.%f"))        # timestemp_srv
                , f"{self._batch_sn}"                         # batch_sn_srv
                , f"'{aUId}'"                                 # unit_id
                , f"{aPId}"                                   # batch_id_unit
                , f"{aTick}"                                  # tick
                , f"'{aType}'"                                # type
                , f"'{aData}'"]                               # data
    
    @abstractmethod
    def _write(self, aRow):
        raise NotImplementedError("Subclass should implement this!")
    
    def ingest(self, aUId, aBId, aTick, aType, aData):
        fields = self._combine_fields(aUId, aBId, aTick, aType, aData)
        self._write(fields)        
        self._batch_sn += 1
                                  
    
    @abstractmethod
    def close(self):
        raise NotImplementedError("Subclass should implement this!")

## SQLite

In [ ]:
class DbWriterSqlite(DBWriter):     
    def __init__(self, aDbFile):
        super().__init__(self)
                
        self.__conn = sqlite3.connect(aDbFile, check_same_thread=False) 
        self.__cursor = self.__conn.cursor()
    

    def _write(self, aRow):
        try:
            self.__cursor.execute(f"INSERT INTO {DB_TBL_NAME_RAW_DATA} VALUES ({', '.join(aRow)})")
            self.__conn.commit()
        except sqlite3.Error as e:
            print(f"SQLite error: {e}")    

    def close(self):
        self.__conn.close()

## Text file

In [ ]:
class FileWriterTsv(DBWriter):     
    def __init__(self, aFile):
        super().__init__(self)

        self.__conn = open(aFile, 'a', newline = '')       

    
    def _write(self, aRow):
        self.__conn.write('\t'.join(aRow) + '\r\n')  
        self.__conn.flush()

    def close(self):
        self.__conn.close()

# DBWriter factory

In [ ]:

def get_dbwriter(aType, aDevice):
    if aType == DBWRITER_TYPE_TSV:
        return FileWriterTsv(aDevice)        
    elif aType == DBWRITER_TYPE_SQLITE:
        return DbWriterSqlite(aDevice)
    else:
        return None

# Tests

In [ ]:
# dumpper = get_dbwriter(DBWRITER_TYPE_SQLITE, DB_PATH_DEVICE_SQLITE)

In [ ]:
# dumpper.ingest('TestDeviceName', 2026, 20250401, 'DM', '{"AccX":-3.36, "AccY":-6.19, "AccZ":5.18, "GyrX":-0.05, "GyrY":0.06, "GyrY":-0.02, "TmpC":19.10}')

In [ ]:
# dumpper = get_dbwriter(DBWRITER_TYPE_TSV, DB_PATH_DEVICE_TSV)

In [ ]:
# dumpper.close()